# Mini-Projeto Avaliativo – Módulo 1 · Análise de Dados com Python [T8]

**Aluno:** Matheus Wander da Silva · **Turma:** T8

**Objetivo:** fazer uma Análise Exploratória de Dados (AED) da base **Varejo** (Kaggle, `namespaiva/base-varejo`),
diagnosticar e tratar a qualidade dos dados, resumir o comportamento de compra e exportar uma base tratada
(`varejo_limpo.csv`).

**Fluxo do notebook:** leitura → diagnóstico → limpeza (função `tratar_dados`) → comparação antes/depois →
estatísticas → filtro → agrupamentos (função `gerar_resumo`) → exportação → conclusões.

## 1. Importação e configuração

A base original fica em `dados/Base_Varejo_csv.zip` (o zip baixado do Kaggle, com o arquivo `Base Varejo.csv` dentro).
O pandas lê o CSV direto do zip, então não é preciso descompactar.

> **No Google Colab:** rode antes `!git clone https://github.com/OMatheusWander/Miniprojeto_MatheusWander_T8.git`
> e depois `%cd Miniprojeto_MatheusWander_T8`.

In [1]:
import pandas as pd

# Caminhos de entrada e saída (relativos à raiz do repositório)
CAMINHO_ENTRADA = "dados/Base_Varejo_csv.zip"
CAMINHO_SAIDA = "varejo_limpo.csv"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## 2. Leitura e diagnóstico da estrutura (Sprint 1)

O arquivo usa **ponto e vírgula** como separador (padrão de planilhas em português), por isso `sep=";"`.

In [2]:
# Leitura do CSV original (o pandas descompacta o zip automaticamente)
df_bruto = pd.read_csv(CAMINHO_ENTRADA, sep=";")

print(f"Registros (linhas): {df_bruto.shape[0]:,}".replace(",", "."))
print(f"Colunas: {df_bruto.shape[1]}")
df_bruto.head()

Registros (linhas): 830.000
Colunas: 14


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


In [3]:
# Nomes das colunas e tipos de dados inferidos pelo pandas
print(df_bruto.columns.tolist())
df_bruto.dtypes

['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


DATA               str
CO_ID            int64
CL_ID            int64
CL_GENERO          str
CL_EC            int64
CL_FHL           int64
CL_SEG             str
PR_ID            int64
PR_CAT             str
PR_NOME            str
Unnamed: 10    float64
Unnamed: 11    float64
Unnamed: 12    float64
Unnamed: 13    float64
dtype: object

**O que cada linha representa?** Conferindo as chaves:

In [4]:
# Cada compra (CO_ID) pertence a um único cliente e a uma única data?
print("Clientes por compra (máx.):", df_bruto.groupby("CO_ID")["CL_ID"].nunique().max())
print("Datas por compra (máx.):   ", df_bruto.groupby("CO_ID")["DATA"].nunique().max())

# Os atributos do cliente são fixos para cada CL_ID?
for col in ["CL_GENERO", "CL_EC", "CL_FHL", "CL_SEG"]:
    print(f"Valores distintos de {col} por cliente (máx.):", df_bruto.groupby("CL_ID")[col].nunique().max())

print()
print("Clientes distintos:", df_bruto["CL_ID"].nunique())
print("Compras distintas: ", df_bruto["CO_ID"].nunique())
print("Produtos (PR_ID):  ", df_bruto["PR_ID"].nunique())

Clientes por compra (máx.): 1
Datas por compra (máx.):    1


Valores distintos de CL_GENERO por cliente (máx.): 1
Valores distintos de CL_EC por cliente (máx.): 1
Valores distintos de CL_FHL por cliente (máx.): 1


Valores distintos de CL_SEG por cliente (máx.): 1

Clientes distintos: 1000
Compras distintas:  18471
Produtos (PR_ID):   229


**Unidade de análise:** cada linha é **um item (produto) dentro de uma compra**. Uma compra (`CO_ID`) tem vários itens,
sempre do mesmo cliente e da mesma data, e cada cliente tem atributos fixos (gênero, estado civil, filhos e segmento).
Por isso, ao longo da análise, **registros = itens**, que é diferente de **compras** (`CO_ID` distintos) e de **clientes**
(`CL_ID` distintos).

| Coluna original | Significado (inferido dos dados e do nome) |
|---|---|
| `DATA` | data da compra (dd/mm/aaaa) |
| `CO_ID` | identificador da compra |
| `CL_ID` | identificador do cliente |
| `CL_GENERO` | gênero do cliente (F/M) |
| `CL_EC` | estado civil do cliente, **codificado de 1 a 5** (a base não traz o dicionário dos códigos) |
| `CL_FHL` | número de filhos do cliente (0 a 4) |
| `CL_SEG` | segmento do cliente (A, B ou C) |
| `PR_ID` | identificador do produto |
| `PR_CAT` | categoria do produto |
| `PR_NOME` | nome do produto |
| `Unnamed: 10` a `13` | colunas sem nome geradas por `;` sobrando no fim de cada linha |

A base **não tem coluna de preço, valor ou quantidade**. Portanto, não é possível calcular faturamento.

## 3. Diagnóstico de qualidade

A função abaixo é usada **antes e depois** da limpeza, para comparar os resultados.
Ela conta, por coluna: valores ausentes (`NaN`), textos vazios ou só com espaços, o marcador de erro do Excel `#N/D`,
textos com espaço sobrando nas pontas e textos que não estão em caixa alta (o padrão da base).

In [5]:
def diagnosticar(df):
    """Recebe um DataFrame e retorna uma tabela com problemas de qualidade por coluna."""
    linhas = []
    for col in df.columns:
        serie = df[col]
        texto = serie.dropna().astype(str)
        eh_texto = serie.dtype == object or pd.api.types.is_string_dtype(serie)
        linhas.append({
            "coluna": col,
            "tipo": str(serie.dtype),
            "ausentes_NaN": int(serie.isna().sum()),
            "texto_vazio_ou_espacos": int((texto.str.strip() == "").sum()) if eh_texto else 0,
            "marcador_#N/D": int((texto == "#N/D").sum()) if eh_texto else 0,
            "espaco_nas_pontas": int((texto != texto.str.strip()).sum()) if eh_texto else 0,
            "fora_caixa_alta": int((texto != texto.str.upper()).sum()) if eh_texto else 0,
        })
    return pd.DataFrame(linhas).set_index("coluna")

diag_antes = diagnosticar(df_bruto)
diag_antes

,tipo,ausentes_NaN,texto_vazio_ou_espacos,marcador_#N/D,espaco_nas_pontas,fora_caixa_alta
coluna,,,,,,
DATA,str,0,0,0,0,0
CO_ID,int64,0,0,0,0,0
CL_ID,int64,0,0,0,0,0
CL_GENERO,str,0,0,0,0,0
CL_EC,int64,0,0,0,0,0
CL_FHL,int64,0,0,0,0,0
CL_SEG,str,0,0,0,0,0
PR_ID,int64,0,0,0,0,0
PR_CAT,str,0,0,3650,0,0


**Leitura do diagnóstico:**
- As 4 colunas `Unnamed` estão **100% vazias**: são um defeito de formatação do arquivo, não dados.
- Nas 10 colunas reais, **não há valores ausentes (`NaN`)** nem textos vazios. Isso também é um resultado.
- Mas existe ausência "disfarçada": `PR_CAT` e `PR_NOME` têm o marcador **`#N/D`**, um erro de PROCV do Excel.
- `PR_NOME` tem textos fora da caixa alta.

In [6]:
# Quais registros têm #N/D? Todos do mesmo produto?
print(df_bruto.loc[df_bruto["PR_CAT"] == "#N/D", ["PR_ID", "PR_CAT", "PR_NOME"]].value_counts())

# Quais nomes de produto estão fora da caixa alta?
nomes = pd.Series(df_bruto["PR_NOME"].unique())
print("\nFora da caixa alta:", nomes[nomes != nomes.str.upper()].tolist())

# Categorias e valores das colunas de texto curtas
for col in ["PR_CAT", "CL_GENERO", "CL_SEG"]:
    print(f"\n{col}:", df_bruto[col].value_counts().to_dict())

PR_ID  PR_CAT  PR_NOME
107    #N/D    #N/D       3650
Name: count, dtype: int64

Fora da caixa alta: ['REFRIGERANTE LIMaO']

PR_CAT: {'ALIMENTOS': 434767, 'HIGIENE': 155574, 'LIMPEZA': 145754, 'BEBIDAS': 43299, 'PET': 32399, 'ACESSORIOS': 14557, '#N/D': 3650}

CL_GENERO: {'F': 432576, 'M': 397424}



CL_SEG: {'B': 530163, 'C': 232101, 'A': 67736}


Ao revisar a lista completa de nomes de produto, aparece também um erro de digitação: **`HAMBUGUER`** (falta um "R").

In [7]:
print(sorted(df_bruto["PR_NOME"].unique()))

['#N/D', 'ABACATE', 'ABACAXI', 'ABSORVENTE', 'ACHOCOLATADO', 'AGUA SANITARIA', 'ALCOOL', 'ALHO', 'ALIMENTO PARA PASSARO', 'ALMONDEGA', 'AMACIANTE', 'ARROZ', 'ARROZ INTEGRAL', 'ATUM', 'AZEITE', 'AZEITONA', 'BALDE', 'BANANA', 'BATATA', 'BATATA DOCE', 'BIFE DE COXAO MOLE', 'BISCOITO', 'BROCOLIS', 'CAFE', 'CEBOLA', 'CENOURA', 'CERA', 'CHA', 'CHUPETA', 'COGUMELOS', 'CONDICIONADOR', 'COPA SUINA', 'CORACAO DE FRANGO', 'COXA E SOBRECOXA DE FRANGO', 'CREME', 'DANETTE', 'DESENGORDURANTE', 'DESINFETANTE', 'DETERGENTE', 'DOCE', 'ENERGETICO', 'ENXAGUANTE BUCAL', 'ESCOVA DE DENTE', 'FEIJAO', 'FILE DE PEIXE', 'FIO DENTAL', 'FIXADOR', 'FRALDA', 'GEL', 'GRANOLA', 'HAMBUGUER', 'HASTES FLEXIVEIS', 'HIDRATANTE', 'INSETICIDA', 'IOGURTE', 'KETCHUP', 'LATA DE ERVILHA', 'LATA DE MILHO', 'LEITE', 'LEITE CONDENSADO', 'LENCO UMEDECIDO', 'LIMAO', 'LIMPA VIDROS', 'LIMPADOR MULTIUSO', 'LIMPADOR PERFUMADO', 'LINGUICA', 'LUSTRA MOVEIS', 'MACARRAO', 'MACARRAO INSTANTANEO', 'MAMAO', 'MAMAO PAPAYA', 'MANGA', 'MODELADOR'

### 3.1 Linhas duplicadas: são duplicatas confirmadas?

In [8]:
colunas_reais = df_bruto.columns[:10]
dup = df_bruto.duplicated(subset=colunas_reais)
print(f"Linhas idênticas a uma anterior: {dup.sum():,}".replace(",", "."), f"({dup.mean():.1%} da base)")

# Quantas vezes o mesmo produto aparece na mesma compra?
repeticoes = df_bruto.groupby(list(colunas_reais)).size()
print("\nVezes que a mesma linha aparece -> nº de combinações (compra + produto):")
print(repeticoes.value_counts().sort_index())

# Essas repetições estão concentradas em poucas compras (indício de erro de carga) ou espalhadas?
compras_com_rep = df_bruto.loc[dup, "CO_ID"].nunique()
print(f"\nCompras com algum produto repetido: {compras_com_rep:,} de {df_bruto['CO_ID'].nunique():,}".replace(",", "."))

Linhas idênticas a uma anterior: 96.553 (11.6% da base)



Vezes que a mesma linha aparece -> nº de combinações (compra + produto):
1    645670
2     79608
3      7601
4       532
5        33
6         3
Name: count, dtype: int64

Compras com algum produto repetido: 14.566 de 18.471


**Decisão sobre duplicatas: manter.** Uma linha repetida significa que o **mesmo produto aparece mais de uma vez na
mesma compra**. Como a base não tem coluna de quantidade, essa é a forma natural de registrar **mais de uma unidade**
(por exemplo, dois inseticidas no mesmo carrinho). Os números confirmam esse padrão:

- as repetições estão espalhadas pela maioria das compras, e não concentradas em poucos blocos;
- a maioria repete só 2 vezes, e quase nenhuma passa de 4.

Esse comportamento é o de um carrinho de compras, não o de um erro de carga. Como não há nenhuma chave de linha
(por exemplo, um "número do item") que prove a duplicação, **não há duplicatas confirmadas**. Removê-las apagaria cerca
de 11,6% dos itens e subestimaria o volume comprado. Compras diferentes do mesmo cliente também não são tratadas como
duplicatas.

## 4. Limpeza e padronização (Sprints 2 e 3)

A função `tratar_dados` recebe a base bruta e devolve uma **cópia** tratada, sem alterar o original. Cada passo
corresponde a um problema do diagnóstico:

| Problema | Tratamento | Justificativa |
|---|---|---|
| 4 colunas `Unnamed` 100% vazias | remover | não são dados, são `;` sobrando no arquivo |
| nomes de colunas em código | renomear (`CL_FHL` → `qtd_filhos` etc.) | leitura mais clara para quem não conhece a base |
| `PR_NOME` fora da caixa alta (`REFRIGERANTE LIMaO`) | `str.strip().str.upper()` em todas as colunas de texto | padroniza a escrita; o "a" minúsculo era um "Ã" perdido |
| erro de digitação `HAMBUGUER` | corrigir para `HAMBURGUER` | o produto é o mesmo, só a grafia muda |
| `#N/D` em `PR_CAT` e `PR_NOME` (produto 107) | trocar por `NaN` e **preencher** com `NAO IDENTIFICADO` | o item, a compra e o cliente são válidos; remover apagaria compras reais. Não dá para deduzir a categoria sem inventar dado |
| `CL_GENERO` codificado (F/M) | **manter** F/M | o código é consistente (só F e M, sem variações); por extenso o CSV passaria de 50 MB, o limite recomendado pelo GitHub |
| `DATA` como texto | converter para data (`datetime`) | permite filtrar por período |
| linhas repetidas | **manter** | não são duplicatas confirmadas (seção 3.1) |

In [9]:
def tratar_dados(df, valor_ausente="NAO IDENTIFICADO"):
    """Recebe a base bruta e retorna uma cópia tratada.

    Parâmetros:
        df: DataFrame original da base Varejo.
        valor_ausente: texto usado no lugar do marcador de erro '#N/D'.
    Retorno:
        DataFrame tratado (mesmo número de linhas: nenhuma linha válida é descartada).
    """
    tratado = df.copy()

    # 1) Remove colunas totalmente vazias (as 'Unnamed' criadas por ';' no fim da linha)
    tratado = tratado.dropna(axis=1, how="all")

    # 2) Renomeia para nomes descritivos
    tratado = tratado.rename(columns={
        "DATA": "data", "CO_ID": "id_compra", "CL_ID": "id_cliente",
        "CL_GENERO": "genero", "CL_EC": "estado_civil_cod", "CL_FHL": "qtd_filhos",
        "CL_SEG": "segmento", "PR_ID": "id_produto", "PR_CAT": "categoria", "PR_NOME": "produto",
    })

    # 3) Padroniza o texto: sem espaços nas pontas e em caixa alta
    colunas_texto = ["genero", "segmento", "categoria", "produto"]
    for col in colunas_texto:
        tratado[col] = tratado[col].str.strip().str.upper()

    # 4) Corrige o erro de digitação identificado no diagnóstico
    tratado["produto"] = tratado["produto"].replace({"HAMBUGUER": "HAMBURGUER"})

    # 5) '#N/D' é um valor ausente disfarçado: vira NaN e depois recebe um rótulo explícito
    for col in ["categoria", "produto"]:
        tratado[col] = tratado[col].replace("#N/D", pd.NA).fillna(valor_ausente)

    # 6) Data em formato de data (dia/mês/ano)
    tratado["data"] = pd.to_datetime(tratado["data"], format="%d/%m/%Y")

    return tratado


df = tratar_dados(df_bruto)
df.head()

,data,id_compra,id_cliente,genero,estado_civil_cod,qtd_filhos,segmento,id_produto,categoria,produto
0,2019-02-01,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,2019-02-01,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,2019-02-01,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,2019-02-01,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,2019-02-01,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


### 4.1 Comparação antes × depois

In [10]:
diag_depois = diagnosticar(df)

print(f"Registros antes: {len(df_bruto):,} | depois: {len(df):,}".replace(",", "."))
print(f"Colunas antes:   {df_bruto.shape[1]} | depois: {df.shape[1]}")
print(f"Colunas com '#N/D' antes: {(diag_antes['marcador_#N/D'] > 0).sum()} | depois: {(diag_depois['marcador_#N/D'] > 0).sum()}")
print(f"Textos fora da caixa alta antes: {diag_antes['fora_caixa_alta'].sum():,} | depois: {diag_depois['fora_caixa_alta'].sum():,}".replace(",", "."))
print(f"NaN antes (colunas reais): {diag_antes['ausentes_NaN'].iloc[:10].sum()} | depois: {diag_depois['ausentes_NaN'].sum()}")
print()
print("Itens 'NAO IDENTIFICADO':", (df["categoria"] == "NAO IDENTIFICADO").sum())
print("Produtos com 'LIMAO' após padronização:", sorted(df.loc[df["produto"].str.contains("LIMAO"), "produto"].unique()))
diag_depois

Registros antes: 830.000 | depois: 830.000
Colunas antes:   14 | depois: 10
Colunas com '#N/D' antes: 2 | depois: 0
Textos fora da caixa alta antes: 7.320 | depois: 0
NaN antes (colunas reais): 0 | depois: 0

Itens 'NAO IDENTIFICADO': 3650
Produtos com 'LIMAO' após padronização: ['LIMAO', 'REFRIGERANTE LIMAO']


,tipo,ausentes_NaN,texto_vazio_ou_espacos,marcador_#N/D,espaco_nas_pontas,fora_caixa_alta
coluna,,,,,,
data,datetime64[us],0,0,0,0,0
id_compra,int64,0,0,0,0,0
id_cliente,int64,0,0,0,0,0
genero,str,0,0,0,0,0
estado_civil_cod,int64,0,0,0,0,0
qtd_filhos,int64,0,0,0,0,0
segmento,str,0,0,0,0,0
id_produto,int64,0,0,0,0,0
categoria,str,0,0,0,0,0


## 5. Estatísticas básicas (Sprint 4)

**Coluna escolhida: `qtd_filhos` (número de filhos).** É a única medida numérica com significado na base: `estado_civil_cod`
é um código e as colunas `id_` são identificadores, que não servem como medida.

**Atenção à unidade:** `qtd_filhos` é um atributo do **cliente**, mas aparece repetido em cada item comprado. Se as
estatísticas fossem calculadas sobre as 830 mil linhas, clientes que compram mais teriam mais peso. Por isso, a base é
reduzida para **uma linha por cliente** antes do cálculo.

In [11]:
clientes = df.drop_duplicates(subset="id_cliente")

estat_filhos = clientes["qtd_filhos"].describe()
print(f"Unidade analisada: cliente (n = {len(clientes)})")
print(f"Contagem de valores válidos: {int(estat_filhos['count'])}")
print(f"Média de filhos:  {estat_filhos['mean']:.2f}")
print(f"Mínimo:           {int(estat_filhos['min'])}")
print(f"Máximo:           {int(estat_filhos['max'])}")
print()
print("Distribuição (clientes por nº de filhos):")
print(clientes["qtd_filhos"].value_counts().sort_index().to_string())
print(f"\n% de clientes sem filhos: {(clientes['qtd_filhos'] == 0).mean():.1%}")

Unidade analisada: cliente (n = 1000)
Contagem de valores válidos: 1000
Média de filhos:  1.14
Mínimo:           0
Máximo:           4

Distribuição (clientes por nº de filhos):
qtd_filhos
0    527
1    125
2    130
3    121
4     97

% de clientes sem filhos: 52.7%


**Interpretação (unidade = cliente):**
1. A **média é de cerca de 1,14 filho por cliente**, mas ela não descreve o cliente típico: mais da metade dos 1.000
   clientes **não tem filhos** (a mediana é 0). A média sobe por causa dos clientes com 2 a 4 filhos.
2. O **mínimo é 0 e o máximo é 4**. Como nenhum cliente passa de 4, é possível que a base tenha agrupado "4 ou mais" no
   valor 4. Não há dicionário que confirme isso, então o valor é lido literalmente.

*Aprofundamento:* outra medida útil é a de **itens por compra** (unidade = compra), calculada a partir da contagem de linhas.

In [12]:
itens_por_compra = df.groupby("id_compra").size()
print("Itens por compra (unidade = compra):")
print(itens_por_compra.describe().round(1).to_string())

Itens por compra (unidade = compra):
count    18471.0
mean        44.9
std         25.7
min          1.0
25%         23.0
50%         45.0
75%         67.0
max         89.0


## 6. Filtro

**Pergunta:** quem compra os produtos infantis (fralda, papinha, chupeta e mordedor)? Pela lógica, a expectativa
seria encontrar principalmente clientes com filhos. O filtro abaixo isola esses itens para testar essa hipótese, o que
também funciona como teste de coerência da base.

In [13]:
produtos_infantis = ["FRALDA", "PAPINHA INFANTIL", "CHUPETA", "MORDEDOR"]
itens_infantis = df[df["produto"].isin(produtos_infantis)]

print(f"Itens infantis: {len(itens_infantis):,} ({len(itens_infantis) / len(df):.1%} dos itens)".replace(",", "."))

compradores = itens_infantis.drop_duplicates("id_cliente")
sem_filhos = (compradores["qtd_filhos"] == 0).sum()
print(f"Clientes que compraram itens infantis: {len(compradores)}")
print(f"  ...dos quais SEM filhos: {sem_filhos} ({sem_filhos / len(compradores):.1%})")
print(f"% dos itens infantis comprados por clientes sem filhos: {(itens_infantis['qtd_filhos'] == 0).mean():.1%}")

Itens infantis: 29.157 (3.5% dos itens)
Clientes que compraram itens infantis: 1000
  ...dos quais SEM filhos: 527 (52.7%)
% dos itens infantis comprados por clientes sem filhos: 52.7%


**Interpretação:** **todos os 1.000 clientes** compraram algum item infantil no período, e **52,7% deles não têm filhos**,
exatamente a proporção de clientes sem filhos na base inteira. Ou seja, o número de filhos **não muda** a chance de comprar esses
itens. Isso pode indicar compras para terceiros (presentes, netos, sobrinhos) ou, mais provavelmente, que a base é
**sintética/simulada**, com produtos distribuídos de forma aleatória. Essa hipótese pesa nas conclusões, ver a seção
de limitações.

## 7. Agrupamentos

A função `gerar_resumo` agrupa a base por uma coluna e devolve, para cada grupo, três métricas diferentes. Elas não
devem ser confundidas:
- **itens**: número de linhas, ou seja, de produtos registrados;
- **compras**: número de `id_compra` distintos (cestas);
- **clientes**: número de `id_cliente` distintos.

Como a base não tem valores em R$, **nenhuma métrica aqui é faturamento**.

In [14]:
def gerar_resumo(df, coluna):
    """Agrupa a base por `coluna` e retorna itens, compras e clientes distintos por grupo.

    Parâmetros:
        df: base tratada (uma linha = um item de uma compra).
        coluna: nome da coluna usada para agrupar.
    Retorno:
        DataFrame ordenado pelo número de itens, com a participação % nos itens.
    """
    resumo = df.groupby(coluna).agg(
        itens=("id_compra", "size"),
        compras=("id_compra", "nunique"),
        clientes=("id_cliente", "nunique"),
    )
    resumo["pct_itens"] = (resumo["itens"] / resumo["itens"].sum() * 100).round(1)
    resumo["itens_por_compra"] = (resumo["itens"] / resumo["compras"]).round(1)
    return resumo.sort_values("itens", ascending=False)

### 7.1 Itens por categoria de produto

In [15]:
resumo_categoria = gerar_resumo(df, "categoria")
resumo_categoria

,itens,compras,clientes,pct_itens,itens_por_compra
categoria,,,,,
ALIMENTOS,434767,18267,1000,52.4,23.8
HIGIENE,155574,17610,1000,18.7,8.8
LIMPEZA,145754,17475,1000,17.6,8.3
BEBIDAS,43299,14735,1000,5.2,2.9
PET,32399,13495,1000,3.9,2.4
ACESSORIOS,14557,9224,1000,1.8,1.6
NAO IDENTIFICADO,3650,3228,956,0.4,1.1


**O que mede:** `itens` é quantas vezes produtos de cada categoria foram registrados. `compras` é em quantas cestas a
categoria apareceu pelo menos uma vez. `clientes` é quantos clientes compraram a categoria pelo menos uma vez.
Neste agrupamento, `itens_por_compra` é a média de itens da categoria nas compras em que ela aparece.

**ALIMENTOS** concentra mais da metade dos itens, seguida de **HIGIENE** e **LIMPEZA**. Parte da diferença vem do
tamanho do sortimento: categorias com mais produtos cadastrados tendem a ter mais itens. Por isso a participação mostra
o **peso no volume de itens**, não necessariamente a preferência do cliente.

In [16]:
# Contexto para a interpretação: quantos produtos distintos (id_produto) cada categoria tem
df.groupby("categoria")["id_produto"].nunique().sort_values(ascending=False)

categoria
ALIMENTOS           120
HIGIENE              43
LIMPEZA              40
BEBIDAS              12
PET                   9
ACESSORIOS            4
NAO IDENTIFICADO      1
Name: id_produto, dtype: int64

### 7.2 Itens, compras e clientes por gênero

In [17]:
resumo_genero = gerar_resumo(df, "genero")
resumo_genero

,itens,compras,clientes,pct_itens,itens_por_compra
genero,,,,,
F,432576,9615,519,52.1,45.0
M,397424,8856,481,47.9,44.9


**O que mede:** a mesma lógica, agora por gênero do cliente. As mulheres respondem por cerca de 52% dos itens, mas
também são cerca de 52% dos clientes. A diferença no total de itens vem principalmente de **haver mais clientes
mulheres**, e não de cestas maiores: os itens por compra são praticamente iguais entre os gêneros.

In [18]:
# Aprofundamento: o mesmo resumo por segmento, reaproveitando a função
gerar_resumo(df, "segmento")

,itens,compras,clientes,pct_itens,itens_por_compra
segmento,,,,,
B,530163,11843,645,63.9,44.8
C,232101,5136,271,28.0,45.2
A,67736,1492,84,8.2,45.4


## 8. Exportação (Sprint 5)

In [19]:
df.to_csv(CAMINHO_SAIDA, index=False)

# Confere relendo o arquivo gravado
verificacao = pd.read_csv(CAMINHO_SAIDA)
print(f"Registros antes da limpeza: {len(df_bruto):,}".replace(",", "."))
print(f"Registros exportados:       {len(verificacao):,}".replace(",", "."))
print(f"Colunas exportadas:         {verificacao.shape[1]} -> {verificacao.columns.tolist()}")

Registros antes da limpeza: 830.000
Registros exportados:       830.000
Colunas exportadas:         10 -> ['data', 'id_compra', 'id_cliente', 'genero', 'estado_civil_cod', 'qtd_filhos', 'segmento', 'id_produto', 'categoria', 'produto']


### Resumo dos problemas tratados ou mantidos

| Problema | Quantidade encontrada | Decisão |
|---|---|---|
| Colunas vazias (`Unnamed`) | 4 colunas (100% vazias) | removidas |
| Valores ausentes (`NaN`) nas colunas reais | 0 | nada a tratar |
| Textos vazios ou só com espaços | 0 | nada a tratar |
| `#N/D` em categoria e produto | 3.650 itens (produto 107) | preenchidos com `NAO IDENTIFICADO`; linhas mantidas |
| Nome fora da caixa alta (`REFRIGERANTE LIMaO`) | 7.320 itens | padronizado para `REFRIGERANTE LIMAO` |
| Erro de digitação `HAMBUGUER` | 1 nome de produto | corrigido para `HAMBURGUER` |
| Linhas repetidas | 96.553 (11,6%) | **mantidas**: mesmo produto mais de uma vez na cesta, não são duplicatas confirmadas |
| Duplicatas confirmadas | 0 | nenhuma remoção |

Nenhum registro foi removido: a base passou de 830.000 para 830.000 linhas e de 14 para 10 colunas.

## 9. Conclusões

1. **A base tem 830.000 itens, 18.471 compras e 1.000 clientes**, entre jan/2019 e dez/2022. Cada compra tem em
   média **~45 itens** (de 1 a 89), um perfil de "compra do mês".
2. **ALIMENTOS responde por ~52% dos itens**, seguida de **HIGIENE (~19%)** e **LIMPEZA (~18%)**. ACESSORIOS é a
   menor categoria (~2%).
3. **Os clientes têm em média 1,14 filho**, mas **52,7% não têm filhos**. A média não representa o cliente típico.
4. **Mulheres são ~52% dos clientes e ~52% dos itens**, e o tamanho da cesta é praticamente igual entre os gêneros.
   O gênero, sozinho, não diferencia o comportamento de compra nesta base.
5. **O segmento B concentra 64,5% dos clientes** e cerca de 64% dos itens.

**Recomendação:** como ALIMENTOS domina o volume e as cestas são grandes, as ações promocionais e o abastecimento devem
priorizar essa categoria, por exemplo com promoções cruzadas entre ALIMENTOS e HIGIENE/LIMPEZA, que aparecem juntas
na maioria das compras. Antes de segmentar campanhas por gênero ou número de filhos, vale validar se esses atributos
de fato influenciam a compra, porque nesta base eles não mostraram diferença.

**Limitação:** a base **não tem preço nem quantidade**, então não mede faturamento e trata cada linha repetida como uma
unidade. Além disso, o filtro de itens infantis mostrou que eles são comprados por todos os clientes, inclusive
todos os clientes sem filhos, sinal de que os dados podem ser **simulados**. As conclusões descrevem a base, não necessariamente um varejo real.